[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_03/3_stats.ipynb)

# Day 3: Statistics, A/B testing and product (easy)

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (25 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | The fraud alert | easy | 7 |
| Q2 | Is it spam if it says FREE? | easy | 7 |
| Q3 | Switch or stay? | easy | 6 |
| Q4 | Twenty metrics, one winner (review) | easy | 5 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

All real datasets (see DATASETS.md for sources). The setup cell loads them:

- `sms`: 5,574 SMS messages (`label` ham/spam, `text`).

Classic probability questions need no data: solve them on paper first, then check with a simulation (`rng = np.random.default_rng(0)`).

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import numpy as np
import pandas as pd
from scipy import stats as st
pd.set_option("display.width", 120)

_sms = os.path.join(DATA, "sms_spam.tsv")
if not os.path.exists(_sms):  # the UCI file comes as a zip
    import zipfile, io as _io
    _z = urllib.request.urlopen("https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip").read()
    open(_sms, "wb").write(zipfile.ZipFile(_io.BytesIO(_z)).read("SMSSpamCollection"))
sms = pd.read_csv(_sms, sep="\t", header=None, names=["label", "text"], quoting=3)
for _name in ['sms']:
    print(_name, globals()[_name].shape)

---
### Q1. The fraud alert

*easy, about 7 min*

A payments team has a fraud detector. 1% of transactions are fraud. The detector flags 95% of fraud (sensitivity) and wrongly flags 10% of good transactions (false positive rate, so specificity 90%).

1. A transaction is flagged. What is the probability it is really fraud? Compute it with Bayes' rule.
2. A second, independent detector with the same quality also flags it. Update the probability.
3. Check part 1 with a simulation of 1,000,000 transactions (`rng = np.random.default_rng(0)`).

This is the same maths as the classic 'positive medical test for a rare disease' question.

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: a rare event plus an imperfect test, and the question asks P(event | positive). Topic: Bayes' rule with a base rate: `P(F | +) = P(+ | F) P(F) / P(+)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. `P(+) = 0.95 * 0.01 + 0.10 * 0.99` (true positives plus false positives). 2. Posterior = `0.95 * 0.01 / P(+)`. 3. For the second flag, use the first posterior as the new prior and repeat. 4. Simulate: `fraud = rng.random(n) < 0.01`; `flag = np.where(fraud, rng.random(n) < 0.95, rng.random(n) < 0.10)`; answer = `fraud[flag].mean()`.

</details>

<details><summary><b>Solution</b></summary>

```python
def posterior(prior, sens=0.95, fpr=0.10):
    return sens * prior / (sens * prior + fpr * (1 - prior))

p1 = posterior(0.01)
p2 = posterior(p1)                     # yesterday's posterior is today's prior
print(f"P(fraud | 1 flag)  = {p1:.4f}")
print(f"P(fraud | 2 flags) = {p2:.4f}")

rng = np.random.default_rng(0)
n = 1_000_000
fraud = rng.random(n) < 0.01
flag = np.where(fraud, rng.random(n) < 0.95, rng.random(n) < 0.10)
print(f"simulated P(fraud | flag) = {fraud[flag].mean():.4f}   flagged share = {flag.mean():.4f}")

# Output:
# P(fraud | 1 flag)  = 0.0876
# P(fraud | 2 flags) = 0.4769
# simulated P(fraud | flag) = 0.0877   flagged share = 0.1080
```

**Why:** Picture 10,000 transactions: 100 are fraud and 95 of them get flagged; 9,900 are good and 990 of them get flagged. So only 95 of 1,085 flags are fraud, about 8.8%. The false positives win because good transactions are 99 times more common. A second independent flag multiplies the odds by the same likelihood ratio again (`0.95 / 0.10 = 9.5`), which lifts the probability to about 48%. Odds form: posterior odds = prior odds x likelihood ratio, so `1/99 x 9.5 = 0.096` odds, which is 8.8%.

**Say it to a PM:** Only about 9 in 100 flagged transactions are really fraud, because fraud is rare and the detector makes many false alarms. Blocking every flag would hurt about 10 good customers for each fraudster caught; a second check raises the hit rate to about 1 in 2.

**Common mistakes:** Answering 95% (that is `P(flag | fraud)`, not `P(fraud | flag)`). Forgetting the false positives in the denominator. Treating the two detectors as independent when they use the same features (then the second flag adds much less).

**Learn more:** [stats-bayes](https://mahsa-agz.github.io/ds-handbook/#stats-bayes), [stats-probability](https://mahsa-agz.github.io/ds-handbook/#stats-probability)

</details>

---
### Q2. Is it spam if it says FREE?

*easy, about 7 min*

Using the real `sms` data, let `has_free` be 'the message contains the word *free*' (whole word, any case: `sms.text.str.contains(r'\bfree\b', case=False)`).
1. Print `P(spam)`, `P(free | spam)` and `P(free | ham)`.
2. Compute `P(spam | free)` with Bayes' rule from those three numbers, and check it by counting directly.
3. A new product has an inbox where only 1% of messages are spam, and the words behave the same. What is `P(spam | free)` there?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# sms is loaded in the setup
has_free = sms.text.str.contains(r'\bfree\b', case=False)

<details><summary><b>Hint 1</b></summary>

Signal: you know how often a word appears in each class and want the class given the word. Topic: Bayes' rule (the building block of the naive Bayes classifier).

</details>

<details><summary><b>Hint 2</b></summary>

1. `spam = sms.label == 'spam'`. 2. `P(free | spam) = has_free[spam].mean()`, same for ham. 3. `P(spam | free) = P(free|spam) P(spam) / (P(free|spam) P(spam) + P(free|ham) P(ham))`. 4. Direct check: `spam[has_free].mean()`. 5. Part 3: same formula with prior 0.01.

</details>

<details><summary><b>Solution</b></summary>

```python
spam = sms["label"] == "spam"
has_free = sms["text"].str.contains(r"\bfree\b", case=False)

def p_spam_given_free(prior, l_spam, l_ham):
    return l_spam * prior / (l_spam * prior + l_ham * (1 - prior))

p_spam, l_spam, l_ham = spam.mean(), has_free[spam].mean(), has_free[~spam].mean()
print(f"P(spam) {p_spam:.3f}  P(free|spam) {l_spam:.3f}  P(free|ham) {l_ham:.4f}")
print(f"Bayes P(spam|free) {p_spam_given_free(p_spam, l_spam, l_ham):.3f}   direct count {spam[has_free].mean():.3f}")
print(f"inbox with 1% spam: P(spam|free) {p_spam_given_free(0.01, l_spam, l_ham):.3f}")

# Output:
# P(spam) 0.134  P(free|spam) 0.228  P(free|ham) 0.0122
# Bayes P(spam|free) 0.742   direct count 0.742
# inbox with 1% spam: P(spam|free) 0.158
```

**Why:** Bayes' rule just reorganises the same counts, so it must match the direct count (0.742). The likelihood ratio `P(free | spam) / P(free | ham)` is about 19: 'free' is strong evidence. But the posterior also depends on the prior: with only 1% spam, the same word gives only about 16%. A classifier trained on one base rate is miscalibrated on another (you must re-weight the prior). Naive Bayes multiplies such likelihood ratios for many words, assuming the words are independent given the class.

**Say it to a PM:** In this data, a text with the word 'free' is spam about 3 times out of 4. In an inbox where spam is rare (1%), the same word means spam only about 1 time in 6, so we should not auto-delete on that word alone.

**Common mistakes:** Using `str.contains('free')`, which also matches 'freedom' and 'carefree'. Reporting `P(free | spam)` (23%) as the answer. Reusing a probability from one population in another with a different base rate.

**Learn more:** [stats-bayes](https://mahsa-agz.github.io/ds-handbook/#stats-bayes), [ai-nlp-basics](https://mahsa-agz.github.io/ds-handbook/#ai-nlp-basics)

</details>

---
### Q3. Switch or stay?

*easy, about 6 min*

A classic (Monty Hall). A prize is behind one of 3 doors. You pick door 1. The host, who knows where the prize is, always opens a door you did not pick that has no prize, and offers a switch.

1. Use Bayes' rule to compute `P(prize behind door 1 | host opens door 3)` and the same for door 2.
2. Simulate 100,000 games with `rng = np.random.default_rng(0)` and print the win rate of 'always stay' and 'always switch'.
3. Why does the host's knowledge matter?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: new information arrives that is not random (the host chooses). Topic: Bayes' rule with likelihoods `P(host opens 3 | prize at d)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. Priors 1/3 each. 2. Likelihoods of 'host opens 3': prize at 1 -> 1/2 (host picks 2 or 3), prize at 2 -> 1 (forced), prize at 3 -> 0. 3. Normalise. 4. Simulation: you pick door 0; staying wins when `prize == 0`; switching wins otherwise (the host removed the only other empty door).

</details>

<details><summary><b>Solution</b></summary>

```python
prior = np.array([1, 1, 1]) / 3
lik = np.array([1 / 2, 1, 0])          # P(host opens door 3 | prize behind door 1, 2, 3)
post = prior * lik / (prior * lik).sum()
print("posterior for doors 1, 2, 3:", np.round(post, 3).tolist())

rng = np.random.default_rng(0)
prize = rng.integers(0, 3, size=100_000)    # you always pick door 0
stay = (prize == 0).mean()
print(f"stay wins {stay:.3f}   switch wins {1 - stay:.3f}")

# Output:
# posterior for doors 1, 2, 3: [0.333, 0.667, 0.0]
# stay wins 0.332   switch wins 0.668
```

**Why:** Your first pick is right 1/3 of the time and the host cannot change that. When you are wrong (2/3 of the time) the host is forced to show the only other empty door, so switching wins exactly then. In Bayes terms, the host opening door 3 is twice as likely when the prize is behind door 2 (probability 1) as when it is behind door 1 (probability 1/2). If the host opened a random door and it happened to be empty, both remaining doors would be 1/2: the information depends on how the data was generated.

**Say it to a PM:** Switching wins two times out of three. The lesson for product data is the same: what a signal means depends on how it was produced, so always ask who chose what we are looking at.

**Common mistakes:** Saying 50/50 because two doors are left. Ignoring the process that created the evidence (a selection effect, like surveying only users who chose to answer).

**Learn more:** [stats-bayes](https://mahsa-agz.github.io/ds-handbook/#stats-bayes), [stats-probability-puzzles](https://mahsa-agz.github.io/ds-handbook/#stats-probability-puzzles)

</details>

---
### Q4. Twenty metrics, one winner (review)

*easy, about 5 min*

Review of probability. An A/B test has no real effect at all, but the dashboard shows 20 independent metrics, each tested at `alpha = 0.05`.
1. What is the probability that at least one metric shows a 'significant' result? How many false positives do you expect?
2. With the Bonferroni rule (test each metric at `0.05 / 20`), what is that probability now?
3. Check part 1 with a simulation: under no effect a p-value is uniform on [0, 1], so draw 100,000 x 20 uniform numbers.

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: 'at least one' across many independent tries. Topic: complement rule, then expected value of a count (binomial mean `n * p`).

</details>

<details><summary><b>Hint 2</b></summary>

1. `1 - 0.95**20`; expected false positives `20 * 0.05`. 2. `1 - (1 - 0.05/20)**20`. 3. `p = rng.random((100_000, 20))`; `(p < 0.05).any(axis=1).mean()`.

</details>

<details><summary><b>Solution</b></summary>

```python
k, alpha = 20, 0.05
print(f"P(at least one false positive) {1 - (1 - alpha) ** k:.3f}   expected false positives {k * alpha:.1f}")
print(f"with Bonferroni: {1 - (1 - alpha / k) ** k:.4f}")
rng = np.random.default_rng(0)
p = rng.random((100_000, k))
print(f"simulated: {(p < alpha).any(axis=1).mean():.3f}")

# Output:
# P(at least one false positive) 0.642   expected false positives 1.0
# with Bonferroni: 0.0488
# simulated: 0.642
```

**Why:** Each metric alone has a 5% false alarm rate, but the chance that all 20 stay quiet is `0.95^20 = 0.358`, so 64% of null experiments show at least one 'winner'. Bonferroni divides alpha by the number of tests and brings the family-wise error back under 5% (it is conservative when metrics are correlated). In practice: pick one primary metric before the test, treat the other metrics as guardrails or exploration, and correct (Bonferroni or Benjamini-Hochberg) when you must test many.

**Say it to a PM:** If we look at 20 metrics, there is about a 64% chance that one of them looks like a win purely by luck. We should decide the main metric before the test and treat surprise wins on other metrics as ideas to re-test, not as results.

**Common mistakes:** Saying 'each test has 5% error so the dashboard has 5% error'. Using `20 x 0.05 = 1` as a probability (it is an expected count). Choosing the primary metric after looking at the results.

**Learn more:** [stats-probability](https://mahsa-agz.github.io/ds-handbook/#stats-probability), [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_03/4_ai.ipynb)